In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import h5py
import importlib

# Support opening notebooks from the root of the repository
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "src" / "observations.py").is_file():
    raise RuntimeError("Open the this notebook from the repo root or notebooks folder.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.observations import load_spectrum
import src.instrument as instrument

In [2]:
spectrum = load_spectrum(ROOT / "data" / "raw" / "observations" / "pg1048_all.dat")

forest = (spectrum.wavelength >= 1220.0) & (spectrum.wavelength <= 1380.0)

wavelength = spectrum.wavelength[forest]
valid = spectrum.valid[forest]

# NaNs break the plotly line plot, so we need to filter them out
# original spectrum is preserved in the `spectrum` variable

flux_plot = np.where(valid, spectrum.flux[forest], np.nan)
error_plot = np.where(valid, spectrum.error[forest], np.nan)

In [3]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=wavelength,
        y=flux_plot,
        customdata=error_plot,
        mode="lines",
        name="Normalized Flux",
        line=dict(color='#24476B', width=1),
        connectgaps=False,
        hovertemplate=(
            "Wavelength: %{x:.3f} Å"
            "<br>Normalized Flux: %{y:.3f}"
            "<br>Flux Error: %{customdata:.3f}"
            "<extra></extra>"
        ),
    )
)

fig.add_hline(y=1.0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="COS Spectrum of PG1048 - candidate Lyman-alpha Forest Region",
    xaxis_title="Wavelength (Å)",
    yaxis_title="Normalized Flux",
    template="plotly_white",
    height=500,
    xaxis=dict(
        range=[1220, 1380],
        rangeslider=dict(visible=True, thickness=0.12)
    ),
)

fig.show(config={"scrollZoom": True})

In [4]:
overview = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.08,
)

# Reuse the flux trace from the first figure.
overview.add_trace(fig.data[0], row=1, col=1)

overview.add_trace(
    go.Scatter(
        x=wavelength,
        y=error_plot,
        mode="lines",
        name="Reported uncertainty",
        line=dict(color="#B45309", width=1),
        connectgaps=False,
        hovertemplate=(
            "Wavelength: %{x:.3f} Å"
            "<br>Reported error: %{y:.4f}"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

overview.add_hline(
    y=1.0, line_dash="dash", line_color="gray", row=1, col=1
)

overview.update_yaxes(title_text="Normalized flux", row=1, col=1)
overview.update_yaxes(
    title_text="Reported 1σ error",
    rangemode="tozero",
    row=2,
    col=1,
)
overview.update_xaxes(
    title_text="Observed wavelength [Å]", row=2, col=1
)
overview.update_xaxes(range=[1220.0, 1380.0])

overview.update_layout(
    title="pg1048_all.dat — spectrum and uncertainty",
    template="plotly_white",
    height=650,
    showlegend=False,
)

overview.show(config={"scrollZoom": True})

median_error = np.nanmedian(error_plot)
print(f"Median normalized uncertainty: {median_error:.4f}")
print(f"Approximate continuum S/N: {1.0 / median_error:.1f} per pixel")

Median normalized uncertainty: 0.0700
Approximate continuum S/N: 14.3 per pixel


In [5]:
C_KMS = 299792.458
LOS_LENGTH_KMS = 2500.0

usable_indices = np.flatnonzero(forest & spectrum.valid)

if usable_indices.size < 2:
    raise ValueError("Not enough usable pixels in the selected range.")

typical_spacing = np.median(
    np.diff(spectrum.wavelength[forest])
)

# Split at excluded rows or unusually large wavelength jumps.
breaks = (
    (np.diff(usable_indices) > 1)
    | (
        np.diff(spectrum.wavelength[usable_indices])
        > 3.0 * typical_spacing
    )
)

split_positions = np.flatnonzero(breaks) + 1
segments = np.split(usable_indices, split_positions)

rows = []

for number, indices in enumerate(segments, start=1):
    wave = spectrum.wavelength[indices]

    # Use pixel-center endpoints for a conservative coverage estimate.
    span_kms = C_KMS * np.log(wave[-1] / wave[0])
    full_los = int(np.floor(span_kms / LOS_LENGTH_KMS))

    rows.append({
        "segment": number,
        "start_A": wave[0],
        "end_A": wave[-1],
        "pixels": len(wave),
        "span_kms": span_kms,
        "full_LOS": full_los,
    })

coverage = pd.DataFrame(rows)

display(
    coverage.round({
        "start_A": 2,
        "end_A": 2,
        "span_kms": 1,
    })
)

print(f"Total full simulation-length chunks: {coverage['full_LOS'].sum()}")

,segment,start_A,end_A,pixels,span_kms,full_LOS
0,1,1220.02,1300.99,2711,19264.5,7
1,2,1306.91,1379.99,2447,16312.7,6


Total full simulation-length chunks: 13


In [6]:
lsf_path = (
    ROOT / "data" / "reference" / "cos_lsf"
    / "aa_LSFTable_G130M_1291_LP1_cn.dat"
)

lsf_wavelengths, lsf_kernels = instrument.load_cos_lsf(lsf_path)

print(f"Wavelength range: {lsf_wavelengths[0]:.0f}–{lsf_wavelengths[-1]:.0f} Å")
print(f"Kernel array shape: {lsf_kernels.shape}")

np.testing.assert_allclose(
    lsf_kernels.sum(axis=0),
    1.0,
    rtol=0.0,
    atol=1e-12,
)

Wavelength range: 1134–1430 Å
Kernel array shape: (321, 57)


In [7]:
TARGET_WAVELENGTH_A = 1300.0
DISPERSION_A_PER_PIXEL = 0.00997  # Nominal G130M dispersion
GAUSSIAN_SIGMA_KMS = 7.96
C_KMS = 299792.458

# Select the closest tabulated wavelength.
index = int(np.argmin(np.abs(lsf_wavelengths - TARGET_WAVELENGTH_A)))
lsf_wave = float(lsf_wavelengths[index])
lsf_kernel = lsf_kernels[:, index]

# Convert native LSF pixel offsets to approximate velocity offsets.
pixel_offsets = np.arange(lsf_kernel.size) - lsf_kernel.size // 2
dv_lsf = C_KMS * DISPERSION_A_PER_PIXEL / lsf_wave
lsf_velocity = pixel_offsets * dv_lsf

# Evaluate the previous Gaussian approximation on the same grid.
gaussian_kernel = np.exp(
    -0.5 * (lsf_velocity / GAUSSIAN_SIGMA_KMS) ** 2
)
gaussian_kernel /= gaussian_kernel.sum()

lsf_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Line core", "Wings — logarithmic scale"],
)

profiles = [
    ("COS LP1", lsf_kernel, "#24476B"),
    ("Gaussian σ = 7.96 km/s", gaussian_kernel, "#B45309"),
]

for label, kernel, color in profiles:
    density = kernel / dv_lsf

    for column in (1, 2):
        # Zero values cannot be displayed on a logarithmic axis.
        plotted = density if column == 1 else np.where(
            density > 0.0, density, np.nan
        )

        lsf_fig.add_trace(
            go.Scatter(
                x=lsf_velocity,
                y=plotted,
                mode="lines",
                name=label,
                legendgroup=label,
                showlegend=(column == 1),
                line=dict(color=color, width=2),
                connectgaps=False,
            ),
            row=1,
            col=column,
        )

lsf_fig.update_xaxes(title_text="Velocity offset [km/s]")
lsf_fig.update_xaxes(range=[-60, 60], row=1, col=1)
lsf_fig.update_xaxes(
    range=[lsf_velocity[0], lsf_velocity[-1]], row=1, col=2
)
lsf_fig.update_yaxes(title_text="LSF density [(km/s)⁻¹]")
lsf_fig.update_yaxes(type="log", range=[-7, -1], row=1, col=2)

lsf_fig.update_layout(
    title=f"LP1 / assumed G130M 1291 — LSF at {lsf_wave:.0f} Å",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", y=-0.25),
)
lsf_fig.show(config={"scrollZoom": True})

print(f"Native LSF sampling: {dv_lsf:.3f} km/s per pixel")

Native LSF sampling: 2.299 km/s per pixel


In [8]:
importlib.reload(instrument)

dv_sim = 2500.0 / 2499
kernel_sim = instrument.resample_lsf(
    lsf_kernel,
    dv_in=dv_lsf,
    dv_out=dv_sim,
)

print(f"Simulation sampling: {dv_sim:.6f} km/s per pixel")
print(f"Kernel length: {kernel_sim.size}")
print(f"Kernel sum: {kernel_sim.sum():.12f}")

np.testing.assert_allclose(
    kernel_sim.sum(), 1.0, rtol=0.0, atol=1e-12
)
assert np.all(kernel_sim >= 0.0)
assert kernel_sim.size % 2 == 1

Simulation sampling: 1.000400 km/s per pixel
Kernel length: 739
Kernel sum: 1.000000000000


In [9]:
importlib.reload(instrument)

# Two diagnostic inputs: a flat continuum and a varying spectrum.
probe = np.vstack([
    np.ones(2499),
    np.linspace(0.0, 1.0, 2499),
])

smoothed, dv_out = instrument.apply_cos_lsf(
    probe,
    dv_sim=dv_sim,
    kernel=kernel_sim,
)

assert smoothed.shape == probe.shape
assert dv_out == dv_sim

# A flat continuum should remain flat.
np.testing.assert_allclose(
    smoothed[0], probe[0], rtol=0.0, atol=1e-12
)

# Each sightline should retain its own mean flux.
np.testing.assert_allclose(
    smoothed.mean(axis=-1),
    probe.mean(axis=-1),
    rtol=0.0,
    atol=1e-12,
)

print("Convolution checks passed.")

Convolution checks passed.


In [10]:
MOCK_MODEL = "EX0"
MOCK_LOS = 9000

calibration_path = (
    ROOT / "outputs" / "uvb_mean_flux" / "calibration"
    / "z0_b15_seed42.json"
)

with calibration_path.open() as handle:
    calibration = json.load(handle)

if calibration["redshift"] != 0.0:
    raise ValueError("This example assumes z = 0")

tau_scale = float(calibration["results"][MOCK_MODEL]["tau_scale"])

if not np.isfinite(tau_scale) or tau_scale <= 0.0:
    raise ValueError("Expected a positive, finite optical-depth scale")

mock_path = ROOT / "data" / "raw" / f"{MOCK_MODEL}_spectra.hdf5"

# Read only the selected sightline.
with h5py.File(mock_path, "r") as handle:
    tau_mock = np.asarray(
        handle[calibration["tau_key"]][MOCK_LOS, :],
        dtype=np.float64,
    )

if tau_mock.ndim != 1 or tau_mock.size < 2:
    raise ValueError("Expected one sightline with at least two pixels")

if not np.all(np.isfinite(tau_mock)) or np.any(tau_mock < 0.0):
    raise ValueError("Optical depth must be finite and nonnegative")

flux_native = np.exp(-tau_scale * tau_mock)
dv_mock = 2500.0 / flux_native.size
velocity_mock = np.arange(flux_native.size) * dv_mock

# Prepare the LSF using this sightline's actual pixel spacing.
kernel_mock = instrument.resample_lsf(
    lsf_kernel,
    dv_in=dv_lsf,
    dv_out=dv_mock,
)

flux_cos, _ = instrument.apply_cos_lsf(
    flux_native,
    dv_sim=dv_mock,
    kernel=kernel_mock,
)

flux_gaussian, _ = instrument.apply_cos_gaussian(
    flux_native,
    dv_sim=dv_mock,
    sigma_kms=7.96,
    truncate=8.0,
)

np.testing.assert_allclose(
    [flux_cos.mean(), flux_gaussian.mean()],
    flux_native.mean(),
    rtol=0.0,
    atol=1e-12,
)

mock_fig = go.Figure()

for label, values, color, dash in [
    ("Native mock", flux_native, "#808080", "dot"),
    ("Gaussian σ = 7.96 km/s", flux_gaussian, "#B45309", "dash"),
    ("COS LP1", flux_cos, "#24476B", "solid"),
]:
    mock_fig.add_trace(
        go.Scatter(
            x=velocity_mock,
            y=values,
            mode="lines",
            name=label,
            line=dict(color=color, dash=dash, width=1.5),
            hovertemplate=(
                "Velocity: %{x:.1f} km/s"
                "<br>Flux: %{y:.4f}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )

mock_fig.update_layout(
    title=(
        f"{MOCK_MODEL}, sightline {MOCK_LOS} — mean-flux matched"
        f"<br><sup>LP1; assumed G130M/1291; "
        f"single LSF profile at {lsf_wave:.0f} Å</sup>"
    ),
    xaxis_title="Velocity along the simulated sightline [km/s]",
    yaxis_title="Normalized flux",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.01),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.1)),
)

mock_fig.show(config={"scrollZoom": True})
print(f"Mean-flux checks passed. Native mean: {flux_native.mean():.6f}")

Mean-flux checks passed. Native mean: 0.898344


In [11]:
NOISE_SEED = 42

# Estimate the sampling of the spectrum using the adjacent valid pixels.
obs_wave = spectrum.wavelength
usable = forest & spectrum.valid
adjacent = usable[1:] & usable[:-1]
adjacent &= np.diff(obs_wave) < 3.0 * typical_spacing # = np.median(np.diff(spectrum.wavelength[forest]))

observed_dv = C_KMS * np.log(obs_wave[1:][adjacent] / obs_wave[:-1][adjacent])
dv_target = float(np.median(observed_dv))
sigma_pilot = float(np.median(spectrum.error[usable]))

# Preserve the full simulation length of 2500 km/s when choosing the output size. This is a convenient choice for later analysis, but it is not strictly necessary.
length_kms = flux_cos.size * dv_mock
n_pilot = int(np.rint(length_kms / dv_target))

flux_pilot_clean, dv_pilot = instrument.resample_flux(flux_cos, dv_in=dv_mock, n_pixels_out=n_pilot)

# Coordinates at the center of each resampled bin.
velocity_pilot = (np.arange(n_pilot) + 0.5) * dv_pilot

np.testing.assert_allclose(flux_pilot_clean.mean(), flux_cos.mean(), rtol=0.0, atol=1e-12)

# Add Gaussian noise to the flux after convolution and resampling. The noise is independent of the flux value, which is a reasonable approximation for the COS data.
rng = np.random.default_rng(NOISE_SEED)
flux_pilot_noisy = flux_pilot_clean + rng.normal(loc=0.0, scale=sigma_pilot, size=flux_pilot_clean.shape)

noise_fig = go.Figure()

noise_fig.add_trace(
    go.Scatter(
        x=velocity_pilot,
        y=flux_pilot_noisy,
        mode="lines+markers",
        name="Mock spectrum with noise",
        marker=dict(color="#24476B", size=4),
        error_y=dict(
            type="constant",
            value=sigma_pilot,
            color="#9CA3AF",
            thickness=0.6,
            width=0,
        ),
    )
)

noise_fig.add_trace(
    go.Scatter(
        x=velocity_pilot,
        y=flux_pilot_clean,
        mode="lines",
        name="COS-convolved, resampled, noise-free",
        line=dict(color="#B45309", width=2),
    )
)

noise_fig.update_layout(
    title=(
        f"{MOCK_MODEL}, sightline {MOCK_LOS} — noise test"
        f"<br><sup>LP1; assumed G130M/1291 at {lsf_wave:.0f} Å; σ = {sigma_pilot:.4f}</sup>"
        f"continuum S/N ≈ {1.0 / sigma_pilot:.1f} per pixel</sup>"
    ),
    xaxis_title="Velocity along the simulated sightline [km/s]",
    yaxis_title="Normalized flux",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.02),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.1)),
)

noise_fig.show(config={"scrollZoom": True})

print(f"Pixels: {flux_cos.size} → {n_pilot} (resampled)")
print(f"Output pixel spacing: {dv_pilot:.6f} km/s")
print(f"Noise level: {sigma_pilot:.4f}")
print(f"Continuum S/N: {1.0 / sigma_pilot:.2f} per pixel (approximate)")

Pixels: 2499 → 362 (resampled)
Output pixel spacing: 6.906077 km/s
Noise level: 0.0700
Continuum S/N: 14.29 per pixel (approximate)


In [12]:
lp1_summary_path = ROOT / (
    "outputs/cos_lsf_snr/summary/"
    "z0_b15_s42_four_class_lp1_obs.json"
)
with lp1_summary_path.open() as handle:
    lp1_summary = json.load(handle)

lp1_stages = ("lp1_obs_clean", "lp1_obs_noisy")
lp1_labels = ["LP1: no added noise", "LP1: S/N ≈ 14.3/pixel"]

lp1_accuracy = 100 * np.array([
    lp1_summary["accuracy"][stage]["accuracy"] for stage in lp1_stages
])
lp1_low = 100 * np.array([
    lp1_summary["accuracy"][stage]["ci_low"] for stage in lp1_stages
])
lp1_high = 100 * np.array([
    lp1_summary["accuracy"][stage]["ci_high"] for stage in lp1_stages
])
lp1_change = lp1_summary["changes"]["lp1_obs_noisy - lp1_obs_clean"]

fig_lp1_accuracy = go.Figure(go.Scatter(
    x=lp1_accuracy,
    y=lp1_labels,
    mode="markers+text",
    text=[f"{value:.2f}%" for value in lp1_accuracy],
    textposition="top center",
    marker=dict(size=13, color=["#0072B2", "#D55E00"]),
    error_x=dict(
        type="data",
        symmetric=False,
        array=lp1_high - lp1_accuracy,
        arrayminus=lp1_accuracy - lp1_low,
        color="#444444",
        thickness=2,
        width=7,
    ),
    customdata=np.column_stack([lp1_low, lp1_high]),
    hovertemplate=(
        "%{y}<br>Accuracy: %{x:.2f}%"
        "<br>95% CI: [%{customdata[0]:.2f}, %{customdata[1]:.2f}]%"
        "<extra></extra>"
    ),
))

fig_lp1_accuracy.add_vline(
    x=25,
    line_dash="dash",
    line_color="gray",
    annotation_text="Chance: 25%",
    annotation_position="top right",
)
fig_lp1_accuracy.update_layout(
    title=dict(text=(
        "LP1 mock pilot: four-class classification"
        "<br><sup>15 sightlines per bundle · "
        "95% paired-bundle bootstrap intervals</sup>"
    )),
    template="plotly_white",
    height=440,
    margin=dict(l=190, r=40, t=90, b=120),
    showlegend=False,
    xaxis=dict(title="Held-out accuracy (%)", range=[0, 100]),
    yaxis=dict(
        categoryorder="array",
        categoryarray=lp1_labels,
        autorange="reversed",
    ),
)
fig_lp1_accuracy.add_annotation(
    x=0,
    y=-0.30,
    xref="paper",
    yref="paper",
    xanchor="left",
    yanchor="top",
    showarrow=False,
    align="left",
    text=(
        f"Noisy − clean: <b>{lp1_change['difference_pp']:+.2f} pp</b>"
        f" (95% CI [{lp1_change['ci_low_pp']:+.2f}, "
        f"{lp1_change['ci_high_pp']:+.2f}] pp)<br>"
        "Provisional G130M/1291 at 1300 Å; Gaussian noise σ = 0.07."
    ),
)
fig_lp1_accuracy.show()

In [13]:
lp1_runs_root = ROOT / "outputs/cos_lsf_snr/runs"
lp1_run_suffix = "z0_b15_s42/matched/EX0_EX1_EX2_EX3"

fig_lp1_cm = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("No added noise", "S/N ≈ 14.3/pixel"),
    horizontal_spacing=0.16,
)

for column, stage in enumerate(lp1_stages, start=1):
    metrics_path = (
        lp1_runs_root / stage / lp1_run_suffix / "metrics.json"
    )
    with metrics_path.open() as handle:
        metrics = json.load(handle)

    labels = metrics["models"]
    counts = np.asarray(metrics["confusion_matrix"], dtype=int)
    row_totals = counts.sum(axis=1, keepdims=True)

    if np.any(row_totals == 0):
        raise ValueError(f"{stage}: a true class has no test examples")

    percentages = 100.0 * counts / row_totals

    fig_lp1_cm.add_trace(
        go.Heatmap(
            z=percentages,
            x=labels,
            y=labels,
            coloraxis="coloraxis",
            customdata=counts,
            texttemplate="%{z:.1f}%",
            hovertemplate=(
                "True model: %{y}<br>"
                "Predicted model: %{x}<br>"
                "Count: %{customdata}<br>"
                "Within true class: %{z:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )
    fig_lp1_cm.update_xaxes(
        title_text="Predicted model",
        row=1,
        col=column,
    )
    fig_lp1_cm.update_yaxes(
        title_text="True model",
        autorange="reversed",
        row=1,
        col=column,
    )

    print(f"\n{stage} — label order: {labels}")
    print(counts)

fig_lp1_cm.update_layout(
    title="LP1 pilot: confusion matrices",
    template="plotly_white",
    height=470,
    margin=dict(l=70, r=100, t=90, b=70),
    coloraxis=dict(
        colorscale="Blues",
        cmin=0,
        cmax=100,
        colorbar=dict(title="Within true<br>class (%)"),
    ),
)
fig_lp1_cm.show()


lp1_obs_clean — label order: ['EX0', 'EX1', 'EX2', 'EX3']
[[33 42 30 29]
 [24 70 16 24]
 [23 17 76 18]
 [14 13  9 98]]

lp1_obs_noisy — label order: ['EX0', 'EX1', 'EX2', 'EX3']
[[42 37 32 23]
 [28 31 42 33]
 [32 27 57 18]
 [24 30 22 58]]


In [14]:
STRONG_FLUX_MAX = 0.85
HALF_WINDOW_KMS = 250.0
MAX_CANDIDATES = 8
C_KMS = 299792.458

obs_wave = spectrum.wavelength
obs_flux = spectrum.flux
obs_error = spectrum.error

obs_forest = (obs_wave >= 1220.0) & (obs_wave <= 1380.0)
obs_usable = obs_forest & spectrum.valid
obs_forest_wave = obs_wave[obs_forest]
obs_spacing = np.median(np.diff(obs_forest_wave))

candidate_pool = np.flatnonzero(
    obs_usable & (obs_flux < STRONG_FLUX_MAX)
)
candidate_pool = candidate_pool[
    np.argsort(obs_flux[candidate_pool], kind="stable")
]

strong_centers = []

for index in candidate_pool:
    # Consider the deepest pixels first; keep separate windows.
    overlaps = any(
        abs(C_KMS * np.log(obs_wave[index] / obs_wave[other]))
        < 2 * HALF_WINDOW_KMS
        for other in strong_centers
    )
    if overlaps:
        continue

    left, right = obs_wave[index] * np.exp(
        np.array([-HALF_WINDOW_KMS, HALF_WINDOW_KMS]) / C_KMS
    )

    if left < obs_forest_wave[0] or right > obs_forest_wave[-1]:
        continue

    window = (obs_wave >= left) & (obs_wave <= right)

    if window.sum() < 3 or not np.all(obs_usable[window]):
        continue

    if np.any(np.diff(obs_wave[window]) > 3 * obs_spacing):
        continue

    strong_centers.append(int(index))

    if len(strong_centers) == MAX_CANDIDATES:
        break

if not strong_centers:
    raise ValueError("No strong-trough windows have complete valid coverage")

strong_candidate_table = pd.DataFrame({
    "center_A": obs_wave[strong_centers],
    "flux_at_center": obs_flux[strong_centers],
    "error_at_center": obs_error[strong_centers],
})

strong_candidate_table

,center_A,flux_at_center,error_at_center
0,1222.6501,0.0020,0.0111
1,1334.4863,0.0030,0.0181
2,1260.3870,0.0134,0.0132
3,1287.7261,0.0622,0.0402
4,1344.4957,0.2692,0.0482
5,1253.6045,0.3281,0.0352
6,1285.3358,0.3842,0.0621
7,1258.1760,0.3931,0.0365


In [16]:
fig_strong = go.Figure()
window_titles = []
y_low, y_high = 0.0, 1.0

for number, index in enumerate(strong_centers):
    center = obs_wave[index]
    velocity = C_KMS * np.log(obs_wave / center)
    window = np.abs(velocity) <= HALF_WINDOW_KMS

    window_flux = obs_flux[window]
    window_error = obs_error[window]

    y_low = min(y_low, float(np.min(window_flux - window_error)))
    y_high = max(y_high, float(np.max(window_flux + window_error)))

    window_titles.append(
        f"Strong-trough inspection: {center:.4f} Å"
        "<br><sup>Center = selected flux minimum; line identification pending</sup>"
    )

    fig_strong.add_trace(go.Scatter(
        x=velocity[window],
        y=window_flux,
        mode="lines+markers",
        line=dict(color="#0072B2", width=1.5),
        marker=dict(size=3),
        error_y=dict(
            type="data",
            array=window_error,
            visible=True,
            color="#999999",
            thickness=0.7,
            width=0,
        ),
        customdata=np.column_stack([
            obs_wave[window],
            window_error,
        ]),
        hovertemplate=(
            "λ: %{customdata[0]:.4f} Å<br>"
            "Velocity: %{x:.1f} km/s<br>"
            "Flux: %{y:.3f}<br>"
            "Reported error: %{customdata[1]:.3f}"
            "<extra></extra>"
        ),
        connectgaps=False,
        visible=(number == 0),
    ))

padding = 0.05 * (y_high - y_low)
shared_y_range = [y_low - padding, y_high + padding]
velocity_range = [-HALF_WINDOW_KMS, HALF_WINDOW_KMS]

buttons = []
for number, index in enumerate(strong_centers):
    visibility = [False] * len(strong_centers)
    visibility[number] = True

    buttons.append(dict(
        label=f"{number + 1}: {obs_wave[index]:.4f} Å",
        method="update",
        args=[
            {"visible": visibility},
            {
                "title.text": window_titles[number],
                "xaxis.range": velocity_range,
                "yaxis.range": shared_y_range,
            },
        ],
    ))

fig_strong.add_hline(y=1.0, line_color="gray", line_width=1)
fig_strong.add_hline(
    y=STRONG_FLUX_MAX,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"F = {STRONG_FLUX_MAX:.2f}",
    annotation_position="bottom right",
)
fig_strong.add_vline(
    x=0,
    line_dash="dot",
    line_color="gray",
    opacity=0.5,
)

fig_strong.update_layout(
    title=dict(text=window_titles[0], x=0.02, y=0.95),
    template="plotly_white",
    height=500,
    margin=dict(l=70, r=40, t=125, b=65),
    showlegend=False,
    xaxis=dict(
        title="Velocity relative to selected center (km/s)",
        range=velocity_range,
    ),
    yaxis=dict(title="Normalized flux", range=shared_y_range),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
    )],
)
fig_strong.show()